In [1]:
from pyspark.sql.functions import col
from pathlib import Path
import sys
import os
from pyspark.sql import SparkSession
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "spark"))

from spark.framework.transformation.silver_builder import build_silver
from spark.framework.persistence.persist_silver import persist_silver
from spark.framework.pipeline.pipeline_context import PipelineContext
from spark.common.config_loader import load_config
from spark.framework.metadata.entity_config_loader import load_entity_config
from spark.framework.spark.spark_session import create_spark_session
from spark.framework.logging.logger import get_logger

from spark.framework.transformation.silver_builder import build_silver
from spark.framework.persistence.persist_silver import persist_silver
from spark.framework.pipeline.pipeline_context import PipelineContext

import time
def replay(
        spark,
        dlq_table,
        silver_table,
        entity_config,
        env_config,
        entity_name,
        logger
):

    logger.info("Reading pending replay records")

    dlq_df = (
        spark.read
        .table(dlq_table)
        .filter(col("replay_status") == "PENDING")
    )

    replay_df = (
        dlq_df
        .select(
            "event_id",
            "key",
            "raw_payload",
            "topic",
            "partition",
            "offset",
            "timestamp"
            "timestampType"
        )
        .dropDuplicates(["event_id"])
    )

    validated_df = build_silver(
        replay_df,
        entity_config,
        env_config,
        entity_name,
        logger=logger,
        replay=True
    )

    persist_silver(
        validated_df=validated_df,
        context=PipelineContext(
            entity_name=entity_name,
            batch_id=0,
            layer="silver",
            silver_table=silver_table,
            dlq_table=dlq_table,
            execution_mode="REPLAY"
        ),
        logger=logger,
        replay=True
    )

Creating Spark Session...


In [3]:
PENDING_STATUS = "PENDING"
entity_name = 'customer'

    ####################################################
    # Environment
    ####################################################

env = "local"

print(f"Environment : {env}")
print(f"Replay Entity : {entity_name}")

    ####################################################
    # Config
    ####################################################

env_config = load_config(env)

entity_config = load_entity_config(entity_name)

    ####################################################
    # Logger
    ####################################################

logger = get_logger(
        "Replay Job",
        env_config
    )

    ####################################################
    # Spark Session
    ####################################################

spark = create_spark_session(
        app_name="Replay Job",
        env=env,
        logger=logger
    )

logger.info("Spark Session Created")

   

Environment : local
Replay Entity : customer


26/08/19 21:11:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/19 21:11:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


2026-08-19 21:11:18,835 | INFO     | Replay Job | Spark Session Created


In [5]:
####################################################
    # Tables
    ####################################################

catalog_name = env_config["iceberg"]["catalog_name"]

silver_table = (
        f"{catalog_name}.silver.{entity_name}"
    )

dlq_table = (
        f"{catalog_name}.dlq.records"
    )

    ####################################################
    # Read Pending Records
    ####################################################

logger.info("Reading Pending Replay Records")

replay_df = (
        spark.read
            .table(dlq_table)
            .filter(col("entity_name") == entity_name)
            .filter(col("replay_status") == PENDING_STATUS)
            .dropDuplicates(["event_id"])
    )

replay_count = replay_df.count()

logger.info(f"Replay Records Found : {replay_count}")




2026-08-19 21:14:52,327 | INFO     | Replay Job | Reading Pending Replay Records
2026-08-19 21:14:53,765 | INFO     | Replay Job | Replay Records Found : 0


In [6]:
replay_df.show()

26/08/19 21:15:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-----------+-----+---------+------+--------+---------+-------------+-----------+------------+--------+----------+----------+------------+-------------+-----------+-------------+------------------+----------------+
|key|raw_payload|topic|partition|offset|event_id|timestamp|timestampType|entity_name|failed_layer|batch_id|error_type|error_code|error_column|error_message|retry_count|replay_status|rejected_timestamp|replay_timestamp|
+---+-----------+-----+---------+------+--------+---------+-------------+-----------+------------+--------+----------+----------+------------+-------------+-----------+-------------+------------------+----------------+
+---+-----------+-----+---------+------+--------+---------+-------------+-----------+------------+--------+----------+----------+------------+-------------+-----------+-------------+------------------+----------------+



In [7]:
    ####################################################
    # Build Silver
    ####################################################

logger.info("Starting Replay Processing")

    

validated_df = build_silver(
        replay_df,
        entity_config=entity_config,
        env_config=env_config,
        entity_name=entity_name,
        logger=logger,
        replay=True
    )

2026-08-19 21:16:24,904 | INFO     | Replay Job | Starting Replay Processing
2026-08-19 21:16:24,908 | INFO     | Replay Job | Starting Silver Builder...
2026-08-19 21:16:24,908 | INFO     | Replay Job | silver Data Read Successfully from stream file
2026-08-19 21:16:24,908 | INFO     | Replay Job | Silver Data will be written to: gs://insightflowai-data-prod/silver/customer
2026-08-19 21:16:24,909 | INFO     | Replay Job | Extracting Required Fields for Silver Layer Processing...
2026-08-19 21:16:24,970 | INFO     | Replay Job | calling parse_debezium function to extract before, after, op, source, ts_ms from raw_payload
Running Debezium Parser
2026-08-19 21:16:24,992 | INFO     | Replay Job | Debezium Parser Completed
calling map_entity function to convert cdc event into entity-specific silver records
2026-08-19 21:16:24,993 | INFO     | Replay Job | Running Entity Mapper
Running Normalizer
2026-08-19 21:16:25,084 | INFO     | Replay Job | Normalizer Completed
2026-08-19 21:16:25,084 

In [ ]:




    ####################################################
    # Persist Silver
    ####################################################

persist_silver(
        validated_df=validated_df,
        context=PipelineContext(
            entity_name=entity_name,
            batch_id=int(time.time()),
            layer="silver",
            silver_table=silver_table,
            dlq_table=dlq_table,
            execution_mode="REPLAY"
        ),
        logger=logger,
        replay=True
    )

logger.info("Replay Completed Successfully")


In [ ]:
 ####################################################
    # Tables
    ####################################################

catalog_name = env_config["iceberg"]["catalog_name"]

silver_table = (
        f"{catalog_name}.silver.{entity_name}"
    )

dlq_table = (
        f"{catalog_name}.dlq.records"
    )

    ####################################################
    # Read Pending Records
    ####################################################

logger.info("Reading Pending Replay Records")

replay_df = (
        spark.read
            .table(dlq_table)
            .filter(col("entity_name") == entity_name)
            .filter(col("replay_status") == PENDING_STATUS)
            .dropDuplicates(["event_id"])
    )

replay_count = replay_df.count()

logger.info(f"Replay Records Found : {replay_count}")

if replay_count == 0:
    logger.info("No Pending Replay Records Found.")
    spark.stop()

    ####################################################
    # Build Silver
    ####################################################

logger.info("Starting Replay Processing")

    

validated_df = build_silver(
        replay_df,
        entity_config=entity_config,
        env_config=env_config,
        entity_name=entity_name,
        logger=logger,
        replay=True
    )

    ####################################################
    # Persist Silver
    ####################################################

persist_silver(
        validated_df=validated_df,
        context=PipelineContext(
            entity_name=entity_name,
            batch_id=int(time.time()),
            layer="silver",
            silver_table=silver_table,
            dlq_table=dlq_table,
            execution_mode="REPLAY"
        ),
        logger=logger,
        replay=True
    )

logger.info("Replay Completed Successfully")


In [3]:
validated_df.show()

NameError: name 'validated_df' is not defined

In [44]:
(spark.sql("select * from insightflow.dlq.records")).show()

+--------------------+--------------------+--------------------+---------+------+--------------------+--------------------+-------------+-----------+------------+--------+----------+--------------+---------------+--------------------+-----------+-------------+--------------------+--------------------+
|                 key|         raw_payload|               topic|partition|offset|            event_id|           timestamp|timestampType|entity_name|failed_layer|batch_id|error_type|    error_code|   error_column|       error_message|retry_count|replay_status|  rejected_timestamp|    replay_timestamp|
+--------------------+--------------------+--------------------+---------+------+--------------------+--------------------+-------------+-----------+------------+--------+----------+--------------+---------------+--------------------+-----------+-------------+--------------------+--------------------+
|{"schema":{"type"...|{"schema":{"type"...|insightflow.publi...|        0|     9|insightflo

26/08/19 16:22:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 255333 ms exceeds timeout 120000 ms
26/08/19 16:22:33 WARN SparkContext: Killing executors is not supported by current scheduler.
26/08/19 16:22:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [18]:
spark.sql("select event_id from insightflow.dlq.records group by event_id having count(*) =3").show()

+--------------------+
|            event_id|
+--------------------+
|insightflow.publi...|
+--------------------+



In [ ]:
replay_df.show()


+--------------------+--------------------+--------------------+---------+------+--------------------+--------------------+-------------+-----------+------------+--------+----------+--------------+---------------+--------------------+-----------+-------------+--------------------+----------------+
|                 key|         raw_payload|               topic|partition|offset|            event_id|           timestamp|timestampType|entity_name|failed_layer|batch_id|error_type|    error_code|   error_column|       error_message|retry_count|replay_status|  rejected_timestamp|replay_timestamp|
+--------------------+--------------------+--------------------+---------+------+--------------------+--------------------+-------------+-----------+------------+--------+----------+--------------+---------------+--------------------+-----------+-------------+--------------------+----------------+
|{"schema":{"type"...|{"schema":{"type"...|insightflow.publi...|        0|     0|insightflow.publi...|2

In [21]:
temp_view = "replay_updates"

replay_df.createOrReplaceTempView(temp_view)
spark.sql("""
MERGE INTO insightflow.dlq.records t
USING (
    SELECT
        event_id,
        'SUCCESS' AS replay_status
    from insightflow.dlq.records group by event_id having count(*) =3
) s
ON t.event_id = s.event_id
WHEN MATCHED THEN
UPDATE SET
    t.replay_status = "Success"
""")

DataFrame[]

In [23]:
replay_df.printSchema()

replay_df.show(truncate=False)

replay_df.explain(True)

root
 |-- key: string (nullable = true)
 |-- raw_payload: string (nullable = true)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- event_id: string (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: string (nullable = true)
 |-- entity_name: string (nullable = false)
 |-- failed_layer: string (nullable = false)
 |-- batch_id: long (nullable = false)
 |-- error_type: string (nullable = false)
 |-- error_code: string (nullable = false)
 |-- error_column: string (nullable = true)
 |-- error_message: string (nullable = false)
 |-- retry_count: integer (nullable = false)
 |-- replay_status: string (nullable = false)
 |-- rejected_timestamp: timestamp (nullable = false)
 |-- replay_timestamp: timestamp (nullable = true)



+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
replay_df.createOrReplaceTempView(temp_view)

AttributeError: 'NoneType' object has no attribute 'show'

In [4]:
(spark.sql("select * from insightflow.bronze.customer")).show()

+--------------------+--------------------+--------------------+---------+------+--------------------+-------------+
|                 key|               value|               topic|partition|offset|           timestamp|timestampType|
+--------------------+--------------------+--------------------+---------+------+--------------------+-------------+
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    13|2026-07-03 12:51:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    12|2026-07-03 11:57:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    15|2026-07-05 18:11:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    11|2026-07-03 10:46:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    12|2026-07-03 11:57:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|

In [25]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit

from spark.framework.pipeline.pipeline_context import PipelineContext
from spark.framework.validation.constants import SILVER_DROP_COLUMNS, REPLAY_FAILED, REPLAY_SUCCESS
from spark.framework.validation.splitter import split_valid_invalid
from spark.framework.monitoring.summary import summarize
from spark.framework.iceberg.table_manager import write_to_iceberg
from spark.framework.dlq.dlq_writer import (build_dlq_dataframe , write_to_dlq)
from spark.replay.replay_status_updater import update_replay_status
context=PipelineContext(
            entity_name=entity_name,
            batch_id=int(time.time()),
            layer="silver",
            silver_table=silver_table,
            dlq_table=dlq_table,
            execution_mode="REPLAY"
)

logger.info("Persisting Silver Batch")

    ####################################################
    # Split
    ####################################################

valid_df, invalid_df = split_valid_invalid(
        validated_df
    )

    ####################################################
    # Summary
    ####################################################

summarize(
        valid_df=valid_df,
        invalid_df=invalid_df,
        entity_name=context.entity_name,
        batch_id=context.batch_id,
        logger = logger
    )

    ####################################################
    # DLQ
    ####################################################

    #
    # Commit-4
    #


2026-08-19 14:03:47,636 | INFO     | Replay Job | Persisting Silver Batch


26/08/19 14:03:50 WARN DAGScheduler: Broadcasting large task binary with size 1644.6 KiB


2026-08-19 14:03:52,412 | INFO     | Replay Job | ================================================================================
2026-08-19 14:03:52,413 | INFO     | Replay Job | VALIDATION SUMMARY
2026-08-19 14:03:52,413 | INFO     | Replay Job | ================================================================================
2026-08-19 14:03:52,414 | INFO     | Replay Job | Entity             : customer
2026-08-19 14:03:52,414 | INFO     | Replay Job | Batch ID           : 1787128427
2026-08-19 14:03:52,414 | INFO     | Replay Job | Total Records      : 4
2026-08-19 14:03:52,415 | INFO     | Replay Job | Valid Records      : 1
2026-08-19 14:03:52,415 | INFO     | Replay Job | Invalid Records    : 3
2026-08-19 14:03:52,415 | INFO     | Replay Job | Failure Percentage : 75.00%


26/08/19 14:03:52 WARN DAGScheduler: Broadcasting large task binary with size 1324.7 KiB
26/08/19 14:03:54 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


2026-08-19 14:03:55,359 | INFO     | Replay Job | Validation Breakdown
2026-08-19 14:03:55,360 | INFO     | Replay Job | allowed_values           : 4
2026-08-19 14:03:55,360 | INFO     | Replay Job | ================================================================================


26/08/19 14:03:55 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


{'entity_name': 'customer',
 'batch_id': 1787128427,
 'total_records': 4,
 'valid_records': 1,
 'invalid_records': 3,
 'failure_percentage': 75.0,
 'validation_breakdown': {'allowed_values': 4}}

In [26]:

success_df = (
        valid_df
            .select("event_id")
            .distinct()
            .withColumn(
                "replay_status",
                lit(REPLAY_SUCCESS)
            )
            .cache()
        )

failure_df = (
            invalid_df
                .select("event_id")
                .distinct()
                .withColumn(
                    "replay_status",
                    lit(REPLAY_FAILED)
                )
                .cache()
        )
        #update the dlq table status col "replay_status" as succesful using valid df
        

        # Update replay metadata
        #
     
 

In [3]:
spark.sql("select * from insightflow.silver.customer").show()

Py4JJavaError: An error occurred while calling o48.sql.
: org.apache.spark.SparkException: [INTERNAL_ERROR] No active or default Spark session found SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:92)
	at org.apache.spark.SparkException$.internalError(SparkException.scala:96)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$2(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$1(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.active(SparkSession.scala:1030)
	at org.apache.spark.sql.catalyst.parser.extensions.IcebergSparkSqlExtensionsParser.parsePlan(IcebergSparkSqlExtensionsParser.scala:124)
	at org.apache.spark.sql.catalyst.parser.ParserInterface.parsePlanWithParameters(ParserInterface.scala:45)
	at org.apache.spark.sql.catalyst.parser.ParserInterface.parsePlanWithParameters$(ParserIn

In [42]:
if success_df.isEmpty():
        logger.info("No replay status updates required.")
        

    
success_df = success_df.localCheckpoint(eager=True)
temp_view = "replay_updates"
success_df.count()  # Force evaluation to avoid lazy execution issues

success_df.createOrReplaceTempView(temp_view)


26/08/19 14:29:27 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:29:27 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:29:27 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:29:27 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:29:28 WARN DAGScheduler: Broadcasting large task binary with size 1650.3 KiB


In [41]:
spark.sql(f"""merge into {dlq_table} t
            using {temp_view} s
            on t.event_id = s.event_id
            when matched then update set t.replay_status = s.replay_status""").show()

Py4JJavaError: An error occurred while calling o48.sql.
: org.apache.spark.SparkException: [INTERNAL_ERROR] The Spark SQL phase planning failed with an internal error. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:107)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:706)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:719)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$executePhase$1(QueryExecution.scala:330)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:804)
	at org.apache.spark.sql.execution.QueryExecution.executePhase(QueryExecution.scala:329)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$lazyExecutedPlan$1(QueryExecution.scala:285)
	at scala.util.Try$.apply(Try

In [43]:
merge_sql = f"""
        MERGE INTO {dlq_table} t
        USING {temp_view} s

        ON t.event_id = s.event_id

        WHEN MATCHED THEN
        UPDATE SET

            t.retry_count = t.retry_count + 1,
            t.replay_status = s.replay_status,
            t.replay_timestamp = current_timestamp()

    """

logger.info("Updating Replay Status")

spark.sql(merge_sql)

logger.info("Replay Status Updated")

2026-08-19 14:30:00,622 | INFO     | Replay Job | Updating Replay Status


2026-08-19 14:30:08,552 | INFO     | Replay Job | Replay Status Updated


In [30]:
update_replay_status(
            validated_df.sparkSession,
            success_df,
            context.dlq_table,
            logger
        )

26/08/19 14:11:03 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:11:03 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:11:03 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:11:03 WARN DAGScheduler: Broadcasting large task binary with size 1649.9 KiB
26/08/19 14:11:04 WARN DAGScheduler: Broadcasting large task binary with size 1653.7 KiB


2026-08-19 14:11:06,284 | INFO     | Replay Job | Updating Replay Status


Py4JJavaError: An error occurred while calling o48.sql.
: org.apache.spark.SparkException: [INTERNAL_ERROR] The Spark SQL phase planning failed with an internal error. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:107)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:706)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:719)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$executePhase$1(QueryExecution.scala:330)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:804)
	at org.apache.spark.sql.execution.QueryExecution.executePhase(QueryExecution.scala:329)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$lazyExecutedPlan$1(QueryExecution.scala:285)
	at scala.util.Try$.apply(Try

In [39]:
spark.sql("""
SELECT
    event_id,
    replay_status
FROM replay_updates
""").explain(True)

== Parsed Logical Plan ==
'Project ['event_id, 'replay_status]
+- 'UnresolvedRelation [replay_updates], [], false

== Analyzed Logical Plan ==
event_id: string, replay_status: string
Project [event_id#5, replay_status#5482]
+- SubqueryAlias replay_updates
   +- View (`replay_updates`, [event_id#5, replay_status#5482])
      +- Project [event_id#5, SUCCESS AS replay_status#5482]
         +- Deduplicate [event_id#5]
            +- Project [event_id#5]
               +- Filter (isnull(Validation_Error#172) OR (size(Validation_Error#172, false) = 0))
                  +- Project [customer_id#93, customer_name#94, customer_status#95, industry#96, customer_size#97, customer_tier#98, customer_start_date#114, customer_end_date#116, created_timestamp#118, updated_timestamp#120, op#89, ts_ms#122, event_id#5, topic#2, partition#3, offset#4L, timestamp#6, key#86, raw_payload#1, timestampType#7, Validation_Error#172]
                     +- Project [customer_id#93, customer_name#94, customer_status

In [ ]:
update_replay_status(
            validated_df.sparkSession,
            success_df,
            context.dlq_table,
            logger
        )

        #update dlq table status col replay_status to "Failure" using invalid_df
update_replay_status(
            validated_df.sparkSession,
            failure_df,
            context.dlq_table,
            logger
        )


In [ ]:
logger.info("Replay mode - skipping DLQ insert")
valid_df = valid_df.drop(
        *SILVER_DROP_COLUMNS,
        "event_id"
    )

    ####################################################
    # Write valid df to Silver
    ####################################################

write_to_iceberg(
        batch_df=valid_df,
        batch_id=context.batch_id,
        table_name=context.silver_table,
        mode="append",
        logger=logger
    )

logger.info("Silver Persistence Completed")

    